# Lab 2 — Stream a Field-Operations Brief

**Required · 30 minutes · Level 100**

Stream an agent response so the operator sees useful text while the complete
brief is still being generated.

## Learning objectives

- Use `agent.run(..., stream=True)`.
- Distinguish transport streaming from a multi-step workflow.
- Collect chunks for logging or a UI without printing tool payloads.

In [ ]:
import os
import re

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)


def safe_name(value: str, *, max_length: int = 40) -> str:
    value = re.sub(r"[^a-z0-9-]+", "-", value.lower()).strip("-")
    value = re.sub(r"-+", "-", value)
    if not value:
        raise ValueError("Resource namespace must contain a letter or number.")
    return value[:max_length].rstrip("-")


raw_namespace = (
    os.getenv("WORKSHOP_RESOURCE_NAMESPACE")
    or os.getenv("WORKSHOP_TEAM_ID")
    or os.getenv("WORKSHOP_PARTICIPANT_ID")
)
if not raw_namespace:
    raise ValueError(
        "Set WORKSHOP_RESOURCE_NAMESPACE (preferred), WORKSHOP_TEAM_ID, "
        "or WORKSHOP_PARTICIPANT_ID before running workshop labs."
    )

RESOURCE_NAMESPACE = safe_name(raw_namespace)
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL = os.environ["FOUNDRY_MODEL"]

print(f"Namespace: {RESOURCE_NAMESPACE}")
print(f"Model: {MODEL}")

In [ ]:
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

credential = AzureCliCredential()
client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL,
    credential=credential,
)
agent = client.as_agent(
    name=f"field-brief-{RESOURCE_NAMESPACE}",
    instructions=(
        "You write concise shift handover briefs for grid field teams. "
        "Separate observed facts from recommendations. Never claim an action "
        "was completed. Use the headings SITUATION, CHECKS, ESCALATION."
    ),
)

## Participant task

Change one detail in `FIELD_NOTE`, then predict which heading should change.
Do not add personal data or real operational identifiers.

In [ ]:
# TODO(participant): change the synthetic weather or telemetry observation.
FIELD_NOTE = '''
Synthetic training scenario:
- asset TR-104 reports intermittent high-temperature alarms
- wind is increasing near the west service area
- no confirmed outage and no switching instruction has been issued
Prepare a handover brief for the incoming operator.
'''

chunks: list[str] = []
async for update in agent.run(FIELD_NOTE, stream=True):
    text = getattr(update, "text", "") or ""
    if text:
        chunks.append(text)
        print(text, end="", flush=True)
streamed_text = "".join(chunks)

## Deterministic success check

In [ ]:
assert chunks, "No text chunks were received."
assert streamed_text.strip(), "The collected stream is empty."
print(f"\nPASS — collected {len(chunks)} text chunks.")

## Reflection

Streaming improves perceived responsiveness, but it does not make the result
more accurate. Grounding and evaluation are introduced later.

## Optional extension

Record chunk arrival timestamps and compare time-to-first-text with total time.

**Expected artifact:** a streamed, three-section handover brief.